# 02 — Test RAG Pipeline

Build the FAISS index and test retrieval + generation interactively.

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path

## Build FAISS Index

In [ ]:
from approach1_rag.ingestion import build_index

build_index(
    data_path=Path("../data/samples/training_data.jsonl"),
    index_dir=Path("../data/faiss_index"),
)

## Test Retrieval

In [ ]:
from approach1_rag.retriever import Retriever

retriever = Retriever(index_dir=Path("../data/faiss_index"))

In [ ]:
# Test a query
query = "Calculate average salary per department"
results = retriever.search(query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {r['score']:.4f}) ---")
    print(f"Description: {r['description']}")
    print(f"Code preview: {r['pyspark_code'][:100]}...")

## Test Prompt Construction

In [ ]:
from shared.prompt_templates import SYSTEM_PROMPT, build_rag_prompt

prompt = build_rag_prompt(
    description="Calculate average salary per department",
    examples=results,
)

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT ===")
print(prompt)

## Test Full Generation (requires Ollama running)

Start Ollama first: `ollama serve` and `ollama pull deepseek-coder:6.7b-instruct`

In [ ]:
import asyncio
from shared.data_models import GenerationRequest
from approach1_rag.generator import RAGGenerator

generator = RAGGenerator(
    retriever=retriever,
)

request = GenerationRequest(
    description="Calculate average salary per department for active employees"
)

response = await generator.generate(request)
print(f"Model: {response.model}")
print(f"Retrieved examples: {response.retrieved_examples}")
print(f"\nGenerated code:\n{response.pyspark_code}")